## RAG Pipeline: Data Ingestion to Vector DB Pipeline

## Document Loader (PDF Loader)

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\45ary\AppData\Local\Temp\ipykernel_25772\4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [2]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"✅ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"❌ Error: {e}")

    print(f"\nTotal documents loaded: {len(documents)}")
    return all_documents



In [3]:
# # Process all PDFs in the data directory
# all_pdf_documents = process_all_pdfs("../data")
# all_pdf_documents

## Text Splitting (to get chunks)

In [4]:
### Text Splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    # if split_docs:
    #     print(f"\nExample chunk")
    #     print(f"Content: {split_docs[0].page_content[:200]}...")
    #     print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
# chunks = split_documents(all_pdf_documents)
# chunks

## Embedding

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid   
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model -> {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str])->np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns: numpy array of embeddings with shape (len(texts), embedding_data)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}\n")
        return embeddings


In [8]:
# # Initialize embedding manager
# embedding_manager = EmbeddingManager()
# embedding_manager

## VectorStoreDB

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self,collection_name:str = "pdf_documents", persist_directory:str="../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
# vectorstore=VectorStore()
# vectorstore

In [10]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

# Create chunks using Text Splitting
chunks = split_documents(all_pdf_documents)

# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Initialize embedding manager
embedding_manager = EmbeddingManager()

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Initialize vector store
vector_store=VectorStore()

# Store in the vector database
vector_store.add_documents(chunks,embeddings)

Found 3 PDF files to process

Processing: AI_ML_DL.pdf
✅ Loaded 5 pages

Processing: Large_Language_Models.pdf
✅ Loaded 6 pages

Processing: RAG_MCP.pdf
✅ Loaded 7 pages

Total documents loaded: 7
Split 18 documents into 36 chunks
Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13312.79it/s]


Model loaded sucessfully. Embedding dimension: 384
Generatig embeddigs for 36 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.21s/it]


Generated embeddings with shape: (36, 384)

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0
Adding 36 documents to vector store...
Successfully added 36 documents to vector store
Total documents in collection: 36


## Retriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self,vector_store:VectorStore, embedding_manager:EmbeddingManager):
        """Initialize the retriever

        Args:
            vector_store (VectorStore): Wector store containing document embeddings
            embedding_manager (EmbeddingManager): Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """Retrieve relevant documents for a query

        Args:
            query (str): The search query
            tok_k (int, optional): Number of top results to return. Defaults to 5.
            score_threshold (float, optional): Minimum similarity score threshold. Defaults to 0.0.

        Returns:
            List[Dict[str,Any]]: List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        
        try:
            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromeDB uses cosine distance)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")

            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {0}")
            return []

In [15]:
rag_retriever = RAGRetriever(vector_store,embedding_manager)
rag_retriever

In [ ]:
rag_retriever.retrieve(query="What is Large Language Model?")

Retrieving documents for query: 'What is Large Language Model?
Top K: 5, Score threshold: 0.0
Generatig embeddigs for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.21it/s]

Generated embeddings with shape: (1, 384)

Retrieved 1 documents (after filtering)


[{'id': 'doc_cd6428ef_10',
  'content': 'Study notes \x7f concise revision edition\nPage 1\n Large Language Models (LLMs)\n 6 pages \x7f Transformers, tokenization, prompting, generation, fine-tuning and applications\n1. LLM Fundamentals\nDefinition\nA Large Language Model (LLM) is a neural network trained on large text datasets to model language and generate or interpret\ntext. Most modern LLMs use the Transformer architecture.\nCapabilities\nQuestion answering, summarization, rewriting, code generation, translation, classification, extraction and structured output.\nTokens\nText is split into tokens—word pieces, characters or other units. Token counts are not equal to word counts.\nContext window\nThe maximum number of input and generated tokens a model can process in one request.\nParameters\nLearned numerical values inside the model. More parameters do not automatically guarantee better task performance.\nPretraining\nLearns general language patterns, commonly by predicting the nex

In [18]:
rag_retriever.retrieve(query="RAG vs fine-tuning?")

Retrieving documents for query: 'RAG vs fine-tuning?
Top K: 5, Score threshold: 0.0
Generatig embeddigs for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 82.11it/s]

Generated embeddings with shape: (1, 384)

Retrieved 3 documents (after filtering)


[{'id': 'doc_261f865d_19',
  'content': 'Fine-tuning\nContinues training a pretrained model on task- or domain-specific examples to adapt behaviour, style or task performance.\nLoRA\nParameter-efficient fine-tuning that learns small low-rank update matrices while most base weights remain frozen.\nQLoRA\nCombines quantized base weights with LoRA adapters to reduce memory requirements.\nKey takeaway: RAG is usually better for frequently changing facts; fine-tuning is useful for adapting behaviour and task patterns.',
  'metadata': {'file_type': 'pdf',
   'page': 4,
   'doc_index': 19,
   'source_file': 'Large_Language_Models.pdf',
   'author': '(anonymous)',
   'page_label': '5',
   'producer': 'ReportLab PDF Library - (opensource)',
   'total_pages': 6,
   'source': '..\\data\\pdf\\Large_Language_Models.pdf',
   'content_length': 471,
   'moddate': '2026-10-31T17:45:45+00:00',
   'title': 'Large Language Models (LLMs)',
   'trapped': '/False',
   'keywords': '',
   'creator': '(unspecif